# 量化交易系统 - 快速入门研究手册

本 Notebook 演示系统的核心功能：数据获取、回测、策略优化和 AI 预测。

## 目录
1. 环境准备
2. 数据获取与清洗
3. 运行回测
4. 策略对比
5. 参数优化
6. Walk-Forward 验证
7. 信号可视化
8. AI 特征工程

In [ ]:
# 1. 环境准备
import sys
sys.path.insert(0, '../src')

from datetime import datetime
from decimal import Decimal

print('环境准备完成')

In [ ]:
# 2. 数据获取与清洗
from quant_trading.interface.services import generate_demo_bars
from quant_trading.data.pipeline import DataPipeline
from quant_trading.model.instrument import InstrumentId, Exchange

# 生成演示数据
iid = InstrumentId('DEMO', Exchange.SSE)
bars = generate_demo_bars(iid, n=250, start_price=100.0)
print(f'原始数据: {len(bars)} 条K线')
print(f'日期范围: {bars[0].timestamp.date()} ~ {bars[-1].timestamp.date()}')
print(f'价格范围: {min(float(b.low) for b in bars):.2f} ~ {max(float(b.high) for b in bars):.2f}')

# 数据清洗
pipeline = DataPipeline(outlier_threshold=0.3)
clean_bars, stats = pipeline.process(bars)
print(f'\n清洗结果: {stats.summary()}')

In [ ]:
# 3. 运行回测
from quant_trading.interface.services import run_backtest

result = run_backtest(
    strategy_id='dual_ma',
    symbol='600519.SSE',
    start=datetime(2023, 1, 1),
    capital=1_000_000,
    use_demo_data=True,
)

m = result['metrics']
print('=== 双均线策略回测结果 ===')
print(f"总收益率: {m['total_return']*100:+.2f}%")
print(f"夏普比率: {m['sharpe_ratio']:.3f}")
print(f"最大回撤: {m['max_drawdown']*100:.2f}%")
print(f"交易次数: {m['total_trades']}")
print(f"胜率: {m['win_rate']*100:.1f}%")

In [ ]:
# 4. 策略对比
strategies = ['dual_ma', 'rsi', 'macd', 'turtle', 'bollinger', 'grid']
results = {}

for sid in strategies:
    r = run_backtest(
        strategy_id=sid,
        symbol='600519.SSE',
        start=datetime(2023, 1, 1),
        capital=1_000_000,
        use_demo_data=True,
    )
    results[sid] = r

print(f"{'策略':12s} | {'收益率':>8s} | {'Sharpe':>8s} | {'最大回撤':>8s} | {'交易数':>6s} | {'胜率':>6s}")
print('-' * 70)
for sid, r in results.items():
    m = r['metrics']
    print(f"{sid:12s} | {m['total_return']*100:+7.2f}% | {m['sharpe_ratio']:8.3f} | {m['max_drawdown']*100:7.2f}% | {m['total_trades']:6d} | {m['win_rate']*100:5.1f}%")

In [ ]:
# 5. 参数优化
from quant_trading.strategy.optimizer import StrategyOptimizer

optimizer = StrategyOptimizer(
    strategy_id='dual_ma',
    symbol='600519.SSE',
    start=datetime(2023, 1, 1),
    capital=1_000_000,
    use_demo_data=True,
)

opt_results = optimizer.optimize({
    'fast_period': [5, 10, 15, 20],
    'slow_period': [20, 30, 40, 60],
})

print(optimizer.format_table(opt_results, top=10))

best = optimizer.best(opt_results)
if best:
    print(f'\n最优参数: {best.params}')
    print(f'最优收益: {best.total_return*100:+.2f}%')

In [ ]:
# 6. Walk-Forward 验证
from quant_trading.alpha.walkforward import WalkForwardValidator

validator = WalkForwardValidator(
    strategy_id='dual_ma',
    symbol='600519.SSE',
    train_days=120,
    test_days=30,
    capital=1_000_000,
)

wf_result = validator.run(
    start=datetime(2022, 1, 1),
    end=datetime(2024, 6, 1),
    use_demo_data=True,
)

print(WalkForwardValidator.format_report(wf_result))

In [ ]:
# 7. 信号可视化
from quant_trading.analysis.signal_chart import plot_signals, plot_equity_with_drawdown

# 权益与回撤图
fig = plot_equity_with_drawdown(result['equity_curve'], title='双均线策略 - 权益与回撤', show=True)

# K线 + 买卖信号图（需要 plotly）
fig2 = plot_signals(clean_bars[:100], result.get('trades', []), title='策略信号图', show=True)

In [ ]:
# 8. AI 特征工程
from quant_trading.alpha.feature import FeatureEngine

engine = FeatureEngine()
engine.register_defaults()

features = engine.compute_features(clean_bars[:60])
print(f'计算了 {len(features)} 条数据的特征')
if features:
    print(f'特征列: {list(features[0].keys())}')
    print(f'最新特征值: {features[-1]}')

## 下一步

- 使用 `quant data fetch 600519.SSE --provider akshare` 获取真实数据
- 在 Web 仪表盘 `http://127.0.0.1:8888` 中交互式操作
- 编写自定义策略继承 `BarSeriesStrategy`
- 使用 `PredictService` 训练 AI 预测模型